# 01 · Build and clean the font glyph dataset (Kaggle)

Turns Google Fonts into images (one 64×64 picture per character 0-9, A-Z, a-z per font),
then decides which fonts to keep and documents why each removed font was removed.

**Before running:** in the notebook sidebar, turn **Internet on** (needs a phone-verified Kaggle account).
A GPU is not needed for this step.

**After running:** *Save Version → Save & Run All*, then open the version's **Output** and click
**New Dataset** (or **New Version** of your existing `tulisanku-fonts` dataset).
Training notebooks will load it from there.

In [ ]:
REPO_URL = "https://github.com/Alb138/tulisanku.git"

!git clone -q {REPO_URL} /kaggle/working/tulisanku
%cd /kaggle/working/tulisanku
!pip install -q -r requirements.txt

## Step 1 · Build
The Google Fonts download goes to `/tmp` so it is **not** saved as notebook output (~1 GB);
only the finished dataset in `/kaggle/working/fonts` is kept.

In [ ]:
!python scripts/build_font_dataset.py --size 64 --repo-dir /tmp/google-fonts --out-dir /kaggle/working/fonts

## Step 2 · Clean
Removes symbol fonts (barcodes, blocks), fonts whose letters are too faint at 64 px, and near-duplicate fonts.
The raw dataset is not changed: decisions go to `fonts_64_quality.csv` (`reason` is empty for kept fonts).

In [ ]:
!python scripts/clean_font_dataset.py --data-dir /kaggle/working/fonts --min-ink 0.015 --dup-threshold 0.98

## Step 3 · Look at the results
- **Histogram:** fonts with tall loops or swashes sit on the left, symbol fonts make the spike on the right.
- **Removed sheet:** every removed font with its reason. Is anything removed that should stay?
- **Duplicates sheet:** the most similar font pairs, from 1.000 downwards. Where do *identical*
  fonts turn into *different* fonts? That tells us whether `--dup-threshold` is right.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

meta = pd.read_csv("/kaggle/working/fonts/fonts_64_quality.csv").fillna({"reason": ""})
meta["median_glyph_height"].hist(bins=50)
plt.xlabel("median glyph height (fraction of image)"); plt.ylabel("fonts"); plt.show()

kept = meta[meta["reason"] == ""]
display(pd.crosstab(kept["category"], kept["split"], margins=True))

print("Removed fonts:")
display(Image("/kaggle/working/fonts/fonts_64_removed.png"))
print("Most similar font pairs (similarity, family):")
display(Image("/kaggle/working/fonts/fonts_64_duplicates.png"))